# Saved research results — notebook v4.7.0

Code release **5.7.0**. Run from the repository root in the existing environment.
Ordinary execution evaluates the selected fixed expression and displays preserved diagnostic images.
It does not import PySR or start Julia. Test AUC is a selection score.


In [ ]:
# Code release v5.7.0 · notebook artifact v4.7.0
from pathlib import Path
from plot_sr import main as inspect_saved_run

PROJECT_DIR = Path.cwd().resolve()
RUN_SOURCE = PROJECT_DIR / "results/L2HL_1_050/manifest.json"
EXPRESSION_ID = "L2HL_1_050"
DATA_CSV = PROJECT_DIR / "data/masses_exclusions.csv"
SAVE_PLOTS, SHOW_PLOTS = False, True

datasets, selected, evaluations = inspect_saved_run(
    RUN_SOURCE, EXPRESSION_ID, DATA_CSV,
    save_plots=SAVE_PLOTS, show_plots=SHOW_PLOTS,
)
X_train, y_train = datasets["train"]
X_val, y_val = datasets["validation"]


## Three different diagnostics

The four figures above use one fixed selected expression; the six validation subsets involve no refitting.
The two images below came from **eight fresh diagnostic fits per parent run**: four complexity limits
and four training fractions. Those models use PySR's own selection rule, separately from the parent's
test-AUC-selected expression. A single stochastic fit per point does not supply uncertainty estimates.

Original numeric curve arrays were not serialized. The curated manifests preserve the existing images,
eight child-run IDs per parent, and source hashes. New explicitly enabled fits save numeric tables and
lineage in a new ignored directory; they never replace these preserved images.

Hosted PNG/PDF files are optional ignored cache. Missing curve images produce links only; no automatic network access occurs. Run the explicit fetch command to cache them.


In [ ]:
# Version v5.7.0 · fitting occurs only in an explicitly enabled call below.
def plot_pysr_curves(
    model, X_train, y_train, X_val, y_val, *, output_dir, parent_run_id,
    maxsizes=(10, 15, 20, 25), fractions=(0.25, 0.50, 0.75, 1.0), seed=42,
    data_csv=None,
):
    """Fit fresh diagnostic models and retain each child's inputs and scores.

    The seed controls training subsets, not a multithreaded symbolic search.
    Parent test-AUC winner selection is separate from candidate.predict().
    """
    from datetime import datetime, timezone
    import hashlib
    from pathlib import Path
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.metrics import roc_auc_score
    from sklearn.model_selection import train_test_split
    from sklearn.utils.class_weight import compute_sample_weight
    from sr import _write_json_atomic

    for X, labels in ((X_train, y_train), (X_val, y_val)):
        if isinstance(labels, pd.Series) and hasattr(X, "index") and not X.index.equals(labels.index):
            raise ValueError("Feature and label indices must match before curve fitting.")

    y_train = np.asarray(y_train).reshape(-1)
    y_val = np.asarray(y_val).reshape(-1)
    if len(X_train) != len(y_train) or len(X_val) != len(y_val):
        raise ValueError("Feature rows and labels must match.")
    for labels in (y_train, y_val):
        if not np.array_equal(np.unique(labels), [0, 1]):
            raise ValueError("Both datasets must contain both labels: 0 and 1.")
    if (not maxsizes or len(set(maxsizes)) != len(maxsizes)
            or any(isinstance(size, bool) or int(size) != size or size < 1 for size in maxsizes)):
        raise ValueError("Complexity limits must be unique positive integers.")
    if not fractions or len(set(fractions)) != len(fractions) or any(not 0 < value <= 1 for value in fractions):
        raise ValueError("Training fractions must be unique and lie in (0, 1].")

    output_dir = Path(output_dir).resolve()
    output_dir.mkdir(parents=True, exist_ok=False)
    metadata = {
        "version": "0.1.0", "code_release": "5.7.0", "status": "running",
        "created_at_utc": datetime.now(timezone.utc).isoformat(), "parent_run_id": parent_run_id,
        "maxsizes": list(maxsizes), "fractions": list(fractions), "subset_seed": seed,
        "dataset_sha256": hashlib.sha256(Path(data_csv).read_bytes()).hexdigest() if data_csv else None,
        "model_selection": getattr(model, "model_selection", "best"),
        "interpretation": "Fresh diagnostic fits; not evaluation of one fixed parent winner.",
        "validation_row_ids": list(X_val.index) if hasattr(X_val, "index") else list(range(len(y_val))),
        "children": [],
    }
    manifest_path = output_dir / "curve_manifest.json"
    _write_json_atomic(manifest_path, metadata)
    rows = {"validation": [], "learning": []}

    def take_rows(X, indices):
        return X.iloc[indices] if hasattr(X, "iloc") else np.asarray(X)[indices]

    def fit_and_score(kind, value, indices, maxsize):
        child_id = f"{kind}_{len(metadata['children']) + 1}"
        X_sub, y_sub = take_rows(X_train, indices), y_train[indices]
        weights = compute_sample_weight("balanced", y_sub)
        candidate = clone(model).set_params(
            maxsize=maxsize, warm_start=False, run_id=child_id,
            output_directory=str(output_dir / "searches"),
        )
        child = {
            "run_id": child_id, "kind": kind, "x": value, "status": "running",
            "model_parameters": candidate.get_params(),
            "training_row_ids": list(X_sub.index) if hasattr(X_sub, "index") else indices.tolist(),
            "training_labels": (2 * y_sub.astype(float) - 1).tolist(),
            "sample_weights": weights.tolist(),
        }
        metadata["children"].append(child)
        _write_json_atomic(manifest_path, metadata)
        candidate.fit(X_sub, 2 * y_sub.astype(float) - 1, weights=weights)
        train_auc = float(roc_auc_score(y_sub, candidate.predict(X_sub)))
        val_auc = float(roc_auc_score(y_val, candidate.predict(X_val)))
        best = candidate.get_best()
        child.update(status="complete", train_auc=train_auc, validation_auc=val_auc,
                     selected_expression_id=f"{child_id}_0{int(best['complexity'])}",
                     selected_expression={key: best[key] for key in ("complexity", "loss", "equation")})
        rows[kind].append((value, train_auc, val_auc))
        _write_json_atomic(manifest_path, metadata)
        pd.DataFrame(rows[kind], columns=["x", "train_auc", "validation_auc"]).to_csv(
            output_dir / f"{kind}.csv", index=False)

    try:
        indices = np.arange(len(y_train))
        for size in maxsizes:
            fit_and_score("validation", size, indices, size)
        for fraction in fractions:
            subset = indices
            if fraction < 1:
                subset, _ = train_test_split(indices, train_size=fraction, stratify=y_train, random_state=seed)
            fit_and_score("learning", len(subset), subset, model.maxsize)
        arrays = {kind: np.asarray(values) for kind, values in rows.items()}
        fig, axes = plt.subplots(1, 2, figsize=(11, 4))
        for ax, kind, xlabel in zip(axes, ("validation", "learning"),
                                    ("Maximum equation complexity (maxsize)", "Training samples")):
            values = arrays[kind]
            ax.plot(values[:, 0], values[:, 1], "o-", label="Train")
            ax.plot(values[:, 0], values[:, 2], "o-", label="Validation")
            ax.axhline(.5, color="gray", linestyle=":", label="Chance AUC")
            ax.set(title=f"{kind.capitalize()} curve", xlabel=xlabel, ylabel="ROC-AUC", ylim=(0, 1.02))
            ax.grid(alpha=.25)
            ax.legend()
        fig.tight_layout()
        fig.savefig(output_dir / "valid-learn-curve.png", dpi=150)
        metadata["status"] = "complete"
        _write_json_atomic(manifest_path, metadata)
        return fig, arrays
    except Exception as error:
        if metadata["children"] and metadata["children"][-1]["status"] == "running":
            metadata["children"][-1]["status"] = "failed"
        metadata.update(status="failed", error={"type": type(error).__name__, "message": str(error)})
        _write_json_atomic(manifest_path, metadata)
        raise


In [ ]:
# This switch authorizes new searches. Keep False for ordinary saved-result inspection.
RUN_CURVE_FITS = False
CURVE_EXPRESSIONS = ("L2HL_1_050", "L2HL_2_0235")

if RUN_CURVE_FITS:
    from datetime import datetime, timezone
    import uuid
    import matplotlib.pyplot as plt
    from pysr import PySRRegressor

    for expression_id in CURVE_EXPRESSIONS:
        parent_run_id = expression_id.rsplit("_", 1)[0]
        model = PySRRegressor.from_file(run_directory=PROJECT_DIR / "output" / parent_run_id)
        stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
        destination = PROJECT_DIR / "output/curve_diagnostics" / parent_run_id / stamp
        fig, curve_results = plot_pysr_curves(
            model, X_train, y_train, X_val, y_val,
            output_dir=destination, parent_run_id=parent_run_id, data_csv=DATA_CSV,
        )
        if SHOW_PLOTS:
            plt.show()
        else:
            plt.close(fig)
else:
    from IPython.display import Image, Markdown, display
    from fetch_artifacts import cached_artifact, artifact_url

    for expression_id in CURVE_EXPRESSIONS:
        print(f"{expression_id}: saved refitted-model diagnostics; historical numerical tables unavailable.")
        cached = cached_artifact(expression_id, "valid-learn-curve.png", project_dir=PROJECT_DIR)
        if cached is not None:
            display(Image(filename=str(cached)))
        else:
            url = artifact_url(expression_id, "valid-learn-curve.png")
            display(Markdown(f"[View the preserved {expression_id} curve image]({url})"))
            print(f"Optional cache: .venv/bin/python fetch_artifacts.py --expression {expression_id}")
